# ETL de películas y plan mensual

Notebook de uso para el paquete modular `movie_etl`. La configuración de etapas, meses y reglas temáticas se lee de JSON. El conjunto de resultados se publica en JSON y lo consume la app React.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'movie_etl').is_dir():
    PROJECT_DIR = PROJECT_DIR / 'scripts etl python'
sys.path.insert(0, str(PROJECT_DIR.resolve()))
from movie_etl.cli import create_application

app = create_application()
PROJECT_DIR

## Ejecutar el ETL

La ejecución de las etapas SQL modifica `db/movies.duckdb` y los archivos intermedios. Dejá `RUN_ETL = False` para trabajar con tablas existentes y publicar solo el listado JSON.

In [ ]:
RUN_ETL = False
STEPS = None  # Ejemplo: [8, 9]
if RUN_ETL:
    app.run_pipeline(STEPS)

## Consultar y publicar el plan

Los campos del listado son película, puntaje ponderado, géneros, categoría, mes y duración en minutos. Las películas sin una regla de género aparecen como `Sin categoría`; se pueden clasificar agregando excepciones al JSON `movie_dashboard/public/data/category_rules.json`.

In [ ]:
plan = app.publish_plan()
history = app.publish_history()
len(plan), len(history), plan[:3]

In [ ]:
import pandas as pd
from IPython.display import display

plan_df = pd.DataFrame(plan)
display(plan_df.head(30))

## Gráfico exploratorio Plotly

El gráfico de notebook permite hover y zoom. La app React agrega filtros sincronizados entre meses, gráfico y tabla.

In [ ]:
import plotly.express as px

chart_data = plan_df.dropna(subset=['duration_minutes', 'weighted_score'])
fig = px.scatter(chart_data, x='duration_minutes', y='weighted_score', color='category',
                  hover_name='title', hover_data=['month', 'genres'],
                  title='Películas por tema mensual, puntaje y duración',
                  labels={'duration_minutes': 'Duración (minutos)', 'weighted_score': 'Puntaje ponderado'})
fig.show()